# STEERER + MovingDroneCrowd++ — test-only Colab runner

This notebook evaluates the repository's **pretrained image-level STEERER counter** on every frame in the MDC++ test split. It does not instantiate, train, or test SDNet/GD3A video counting.

The durable output is a small per-image `predictions.csv` plus `summary.json`. Density maps are optional and disabled by default because they are not needed for MAE/RMSE or later count-range analysis. Progress is copied to Drive every few images, and an interrupted full test can resume without repeating completed frames.

Recommended order:

1. Select a GPU runtime in Colab.
2. Run all cells once with `ACTION = "smoke"` (the default).
3. Check that three predictions and a summary were saved.
4. Set `ACTION = "evaluate"` and `CONFIRM_FULL_TEST = True`, then rerun the control panel, preflight, action, and result-viewer cells.

The official checkpoint is downloaded automatically and cached in `MyDrive/MDC_STEERER/checkpoints`. The existing dataset archive used by the other notebooks is reused by default, so no duplicate dataset upload is required.


In [6]:
# @title 0. Control panel — normally this is the only cell you edit
from pathlib import Path

# ---------- CURRENT ACTION ----------
ACTION = "inspect"  # @param ["inspect", "smoke", "evaluate"]

# ---------- REPOSITORY AND DATA ----------
REPO_URL = "https://github.com/gabrielxmit10/MovingDroneCrowd.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
REPO_DIR = Path("/content/MDC_STEERER")
DRIVE_PROJECT = Path("/content/drive/MyDrive/MDC_STEERER")

DATA_SOURCE = "archive"  # @param ["archive", "drive_folder", "already_staged"]
DATASET_ARCHIVE = Path("/content/drive/MyDrive/P2PNet_MDC/data/MovingDroneCrowd++.tar")  # @param {type:"string"}
DRIVE_DATASET_FOLDER = Path("/content/drive/MyDrive/P2PNet_MDC/data/MovingDroneCrowd++")  # @param {type:"string"}
DATASET_ROOT = Path("/content/data/MovingDroneCrowd++")
TEST_SPLIT = "test.txt"  # Extended MDC++ test split (not MDC_test.txt)

# ---------- OFFICIAL STEERER CHECKPOINT ----------
CHECKPOINT_SOURCE = "download"  # @param ["download", "drive"]
CHECKPOINT_URL = "https://huggingface.co/fyw1999/MovingDroneCrowd-Weights/resolve/main/GD3A_pre_trained_global_counter_STEERER_MDC%2B%2B_ep_201_mae_13.5_mse_19.1.pth"
DRIVE_CHECKPOINT = DRIVE_PROJECT / "checkpoints/GD3A_pre_trained_global_counter_STEERER_MDC++_ep_201_mae_13.5_mse_19.1.pth"
LOCAL_CHECKPOINT = Path("/content/steerer_checkpoints") / DRIVE_CHECKPOINT.name

# ---------- RUN AND EVALUATION ----------
RUN_NAME = "steerer_mdc_inspect"  # @param {type:"string"}
RUN_DIR = DRIVE_PROJECT / "runs" / RUN_NAME
DEVICE = "cuda:0"  # @param ["cuda:0", "cpu"]
SMOKE_SAMPLES = 3  # @param {type:"integer"}
EVAL_MAX_SAMPLES = 0  # @param {type:"integer"} 0 means the complete test split
FRAME_STRIDE = 1  # @param {type:"integer"} keep 1 for the final result
CONFIRM_FULL_TEST = False  # @param {type:"boolean"}
RESUME_EXISTING = True  # @param {type:"boolean"}
SYNC_EVERY = 10  # @param {type:"integer"}
SEED = 3035  # @param {type:"integer"}

# These reproduce the repository's MDC++ test preprocessing. Changing them changes the experiment.
MAX_LONG_SIDE = 1920  # @param {type:"integer"}
MAX_SHORT_SIDE = 1080  # @param {type:"integer"}
USE_AMP = False  # @param {type:"boolean"} FP32 is the reference setting

# Optional large artifacts. Counts/errors are always saved; density maps are not needed for metrics.
SAVE_DENSITY_MAPS = False  # @param {type:"boolean"}
DENSITY_MAP_LIMIT = 20  # @param {type:"integer"} 0 means every selected image

# Editable only for the post-run analysis table; it does not change model evaluation.
COUNT_BINS = [0, 50, 100, 200, 500, float("inf")]


## 1. Mount Drive and inspect the runtime

Drive folders are created automatically. The only interactive step is Google's normal Drive authorization prompt.


In [7]:
from google.colab import drive
import datetime, json, os, platform, re, shlex, shutil, subprocess, sys, tarfile, time

drive.mount("/content/drive")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
subprocess.run(["nvidia-smi"], check=False)
usage = shutil.disk_usage("/content")
print(f"/content free: {usage.free / 2**30:.1f} GiB")
if DEVICE.startswith("cuda") and shutil.which("nvidia-smi") is None:
    raise RuntimeError("A GPU was requested. In Colab choose Runtime > Change runtime type > GPU.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.13.15
/content free: 188.3 GiB


## 2. Clone the repository and install the minimal STEERER dependencies

The notebook checks that the evaluation backend is present. If this check fails, push this notebook, `evaluate_steerer_mdc.py`, `requirements-colab-steerer.txt`, and `model/density_estimator/STEERER/heads/counting_head.py` to `REPO_URL`/`REPO_REF`, then rerun the cell.


In [ ]:
def clean_repo_url(value):
    value = str(value).strip()
    markdown_link = re.fullmatch(r"\[[^\]]+\]\((https?://[^)]+)\)", value)
    return (markdown_link.group(1) if markdown_link else value).rstrip("\\")

REPO_URL = clean_repo_url(REPO_URL)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
else:
    print("Reusing runtime checkout:", REPO_DIR)
remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
if remote.rstrip("/").removesuffix(".git") != REPO_URL.rstrip("/").removesuffix(".git"):
    raise RuntimeError(f"Existing checkout uses a different origin: {remote}")
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
if REPO_REF == "main":
    subprocess.run(["git", "merge", "--ff-only", "origin/main"], cwd=REPO_DIR, check=True)
required_files = ["evaluate_steerer_mdc.py", "requirements-colab-steerer.txt", "model/density_estimator/STEERER/configs/MDC.py", "model/density_estimator/STEERER/heads/counting_head.py"]
missing_backend = [name for name in required_files if not (REPO_DIR / name).is_file()]
if missing_backend:
    raise RuntimeError(f"The GitHub checkout is missing {missing_backend}. Push the local changes, then rerun.")
head_source = (REPO_DIR / "model/density_estimator/STEERER/heads/counting_head.py").read_text(encoding="utf-8")
if "from .base_head import BaseHead" in head_source:
    raise RuntimeError("The GitHub checkout still has the old MMCV import. Push counting_head.py and rerun this cell.")
# Show the full pip error if installation fails; the Colab Python version may change.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab-steerer.txt"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)

def run_command(command):
    command = [str(value) for value in command]
    rendered = shlex.join(command)
    print("$", rendered)
    with (DRIVE_PROJECT / "action_commands.log").open("a", encoding="utf-8") as handle:
        handle.write(f"{datetime.datetime.now().isoformat()}\t{rendered}\n")
    return subprocess.run(command, cwd=REPO_DIR, check=True)

import torch
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Commit:", commit)
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


## 3. Stage MDC++ on fast runtime storage

The default points at the same archive used by the P2PNet and MPCount notebooks. Copying and extracting it can take several minutes, but inference from `/content` is much faster and more reliable than reading thousands of files directly from mounted Drive.


In [ ]:
DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
if DATASET_ROOT.exists():
    print("Dataset already staged:", DATASET_ROOT)
elif DATA_SOURCE == "archive":
    if not DATASET_ARCHIVE.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {DATASET_ARCHIVE}. Update DATASET_ARCHIVE in the control panel.")
    archive_size = DATASET_ARCHIVE.stat().st_size
    free = shutil.disk_usage("/content").free
    if free < archive_size * 2.2:
        raise RuntimeError(f"Not enough /content disk: free={free/2**30:.1f} GiB, archive={archive_size/2**30:.1f} GiB")
    local_archive = Path("/content") / DATASET_ARCHIVE.name
    extract_root = Path("/content") / f"mdc_extract_{int(time.time())}"
    extract_root.mkdir(parents=True, exist_ok=False)
    print("Copying the archive from Drive to runtime storage...")
    shutil.copy2(DATASET_ARCHIVE, local_archive)
    print("Extracting MDC++...")
    try:
        with tarfile.open(local_archive) as archive:
            try:
                archive.extractall(extract_root, filter="data")
            except TypeError:
                archive.extractall(extract_root)
        candidates = [extract_root / "MovingDroneCrowd++", extract_root]
        source_root = next((candidate for candidate in candidates if (candidate / "frames").is_dir() and (candidate / "annotations").is_dir()), None)
        if source_root is None:
            raise RuntimeError("Archive did not contain the expected MovingDroneCrowd++ layout")
        shutil.move(str(source_root), str(DATASET_ROOT))
    finally:
        local_archive.unlink(missing_ok=True)
        if extract_root.exists():
            shutil.rmtree(extract_root)
elif DATA_SOURCE == "drive_folder":
    if not DRIVE_DATASET_FOLDER.is_dir():
        raise FileNotFoundError(DRIVE_DATASET_FOLDER)
    print("Copying the dataset folder from Drive; an archive is normally faster...")
    shutil.copytree(DRIVE_DATASET_FOLDER, DATASET_ROOT)
elif DATA_SOURCE != "already_staged":
    raise ValueError(f"Unknown DATA_SOURCE={DATA_SOURCE!r}")
required_data = ["frames", "annotations", TEST_SPLIT]
missing_data = [item for item in required_data if not (DATASET_ROOT / item).exists()]
if missing_data:
    raise RuntimeError(f"Staged dataset is incomplete; missing {missing_data} under {DATASET_ROOT}")
print("Dataset ready:", DATASET_ROOT)


## 4. Download/cache the official MDC++ STEERER checkpoint

With `CHECKPOINT_SOURCE = "download"`, no manual upload is needed. The public checkpoint is downloaded once to Drive and reused on future sessions. If automatic download is unavailable, upload that exact `.pth` file to `DRIVE_CHECKPOINT` and select `CHECKPOINT_SOURCE = "drive"`.


In [ ]:
DRIVE_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if CHECKPOINT_SOURCE == "download" and not DRIVE_CHECKPOINT.is_file():
    temporary_download = Path("/content") / (DRIVE_CHECKPOINT.name + ".download")
    print("Downloading official STEERER MDC++ checkpoint...")
    subprocess.run(["wget", "--progress=bar:force:noscroll", "-O", str(temporary_download), CHECKPOINT_URL], check=True)
    if temporary_download.stat().st_size < 1_000_000:
        raise RuntimeError("Downloaded checkpoint is unexpectedly small")
    shutil.copy2(temporary_download, DRIVE_CHECKPOINT)
    temporary_download.unlink()
elif CHECKPOINT_SOURCE not in {"download", "drive"}:
    raise ValueError(f"Unknown CHECKPOINT_SOURCE={CHECKPOINT_SOURCE!r}")
if not DRIVE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {DRIVE_CHECKPOINT}")
LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not LOCAL_CHECKPOINT.is_file() or LOCAL_CHECKPOINT.stat().st_size != DRIVE_CHECKPOINT.stat().st_size:
    print("Copying checkpoint to runtime storage...")
    shutil.copy2(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
print(f"Checkpoint ready: {LOCAL_CHECKPOINT} ({LOCAL_CHECKPOINT.stat().st_size / 2**20:.1f} MiB)")


## 5. Preflight the complete selected split

This parses the split and every relevant annotation file but does not load the model or run inference. For the full extended test set with stride 1, the expected manifest is 2,462 images across 41 clips. If your result differs, stop and check that you staged the intended MDC++ version and `test.txt`.


In [ ]:
from IPython.display import display
import pandas as pd

if ACTION not in {"inspect", "smoke", "evaluate"}:
    raise ValueError(f"Unknown ACTION={ACTION!r}")
if FRAME_STRIDE < 1 or SMOKE_SAMPLES < 1 or EVAL_MAX_SAMPLES < 0:
    raise ValueError("FRAME_STRIDE and SMOKE_SAMPLES must be positive; EVAL_MAX_SAMPLES cannot be negative")
PREFLIGHT_DIR = RUN_DIR / "preflight"
preflight_command = [
    sys.executable, "evaluate_steerer_mdc.py",
    "--data-root", DATASET_ROOT,
    "--split-file", TEST_SPLIT,
    "--output-dir", PREFLIGHT_DIR,
    "--frame-stride", FRAME_STRIDE,
    "--max-samples", 0,
    "--inspect-only",
]
run_command(preflight_command)
inspection = json.loads((PREFLIGHT_DIR / "dataset_inspection.json").read_text(encoding="utf-8"))
display(pd.DataFrame(inspection.items(), columns=["Check", "Value"]))
if FRAME_STRIDE == 1 and inspection["selected_samples"] != 2462:
    print("WARNING: this dataset does not have the expected 2,462 images in the extended test.txt split. Verify the dataset version before final evaluation.")
if ACTION == "evaluate" and not CONFIRM_FULL_TEST:
    print("BLOCKED UNTIL CONFIRMED: set CONFIRM_FULL_TEST=True before the full test action.")
elif ACTION == "inspect":
    print("Inspection complete; no model inference will run.")
else:
    print(f"Ready for ACTION={ACTION!r}.")


## 6. Run the selected action

`smoke` evaluates only the first few test images. `evaluate` uses the complete test split when `EVAL_MAX_SAMPLES = 0` and `FRAME_STRIDE = 1`. Existing compatible CSV rows are reused when resume is enabled.


In [ ]:
if ACTION == "inspect":
    CURRENT_RESULT_DIR = PREFLIGHT_DIR
    print("ACTION='inspect': stopped after dataset preflight.")
else:
    if ACTION == "evaluate" and not CONFIRM_FULL_TEST:
        raise RuntimeError("Set CONFIRM_FULL_TEST=True in the control panel before evaluating the held-out test split.")
    action_name = "smoke" if ACTION == "smoke" else "test"
    max_samples = SMOKE_SAMPLES if ACTION == "smoke" else EVAL_MAX_SAMPLES
    CURRENT_RESULT_DIR = RUN_DIR / action_name
    local_work_dir = Path("/content/steerer_mdc_work") / RUN_NAME / action_name
    command = [
        sys.executable, "evaluate_steerer_mdc.py",
        "--data-root", DATASET_ROOT,
        "--split-file", TEST_SPLIT,
        "--checkpoint", LOCAL_CHECKPOINT,
        "--output-dir", CURRENT_RESULT_DIR,
        "--work-dir", local_work_dir,
        "--device", DEVICE,
        "--frame-stride", FRAME_STRIDE,
        "--max-samples", max_samples,
        "--max-long", MAX_LONG_SIDE,
        "--max-short", MAX_SHORT_SIDE,
        "--seed", SEED,
        "--sync-every", SYNC_EVERY,
        "--density-map-limit", DENSITY_MAP_LIMIT,
        "--save-density-maps" if SAVE_DENSITY_MAPS else "--no-save-density-maps",
        "--use-amp" if USE_AMP else "--no-use-amp",
        "--resume" if RESUME_EXISTING else "--no-resume",
    ]
    run_command(command)
    summary = json.loads((CURRENT_RESULT_DIR / "summary.json").read_text(encoding="utf-8"))
    display(pd.DataFrame([{key: value for key, value in summary.items() if not isinstance(value, (dict, list))}]))
    latest_action = {
        "updated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "action": ACTION,
        "output_dir": str(CURRENT_RESULT_DIR),
        "status": summary["status"],
    }
    (DRIVE_PROJECT / "latest_action.json").write_text(json.dumps(latest_action, indent=2), encoding="utf-8")


## 7. Inspect saved per-image results

This reads only the saved Drive artifacts. It shows the worst individual errors and an editable ground-truth count-range analysis. Changing `COUNT_BINS` and rerunning this cell never reruns the model.


In [ ]:
if ACTION == "inspect":
    print(json.dumps(inspection, indent=2))
else:
    predictions_path = CURRENT_RESULT_DIR / "predictions.csv"
    summary_path = CURRENT_RESULT_DIR / "summary.json"
    if not predictions_path.is_file():
        raise FileNotFoundError(predictions_path)
    predictions = pd.read_csv(predictions_path)
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Saved summary:")
    display(pd.DataFrame([{key: value for key, value in summary.items() if not isinstance(value, (dict, list))}]))
    print("Worst per-image absolute errors:")
    display(predictions.sort_values("absolute_error", ascending=False).head(20))

    if len(COUNT_BINS) < 2 or any(COUNT_BINS[i] >= COUNT_BINS[i + 1] for i in range(len(COUNT_BINS) - 1)):
        raise ValueError("COUNT_BINS must be strictly increasing")
    predictions["ground_truth_range"] = pd.cut(
        predictions["ground_truth_count"], bins=COUNT_BINS, right=False, include_lowest=True
    )
    range_metrics = predictions.groupby("ground_truth_range", observed=False).agg(
        images=("sample_id", "count"),
        ground_truth_mean=("ground_truth_count", "mean"),
        predicted_mean=("predicted_count", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda values: float(values.mean() ** 0.5)),
    ).reset_index()
    print("Metrics by ground-truth crowd-count range:")
    display(range_metrics)
    print("Per-scene diagnostic (not the primary aggregate result):")
    scene_metrics = predictions.groupby("scene").agg(
        images=("sample_id", "count"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda values: float(values.mean() ** 0.5)),
    ).sort_values("mae", ascending=False)
    display(scene_metrics)
    density_files = sorted((CURRENT_RESULT_DIR / "density_maps").glob("*.npz")) if (CURRENT_RESULT_DIR / "density_maps").is_dir() else []
    print(f"CSV size: {predictions_path.stat().st_size / 2**20:.3f} MiB; saved density maps: {len(density_files)}")
    print("Durable result directory:", CURRENT_RESULT_DIR)


## What you must do

- Push this notebook, the evaluation script, the requirements file, and `model/density_estimator/STEERER/heads/counting_head.py` to the configured GitHub repository before opening it in Colab.
- Confirm the default dataset archive path exists in your Drive; otherwise change `DATASET_ARCHIVE`.
- Approve the normal Drive mount prompt and use a GPU runtime.
- Run the smoke test first. After it succeeds, explicitly unlock the complete run with `CONFIRM_FULL_TEST = True`.

You do **not** need to create Drive folders or manually upload the STEERER checkpoint with the default configuration. The notebook handles both. If Colab disconnects during evaluation, rerun setup and the same `evaluate` action; compatible completed rows are retained and skipped.
